# Clean NEI data and create national annual totals

Run from top to bottom with `NEI_econ230a.csv` in the same folder as this notebook (and that folder as the working directory).This notebook requires pandas.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

csv_path = Path("NEI_econ230a.csv")
df = pd.read_csv(csv_path, sep=",")
display(df.head())

## 1. Inspect and standardize the data

In [ ]:
# Inspect original types, missing values, and exact duplicate rows.
df.info()
display(df.isna().sum().rename("missing_values"))
print("Exact duplicate rows:", df.duplicated().sum())

In [ ]:
# Clean text and enforce numeric types; invalid numeric entries raise an error.
for col in ["state", "sector", "pollutant"]:
    df[col] = df[col].astype("string").str.strip().replace("", pd.NA)

for col in ["state_fips", "year", "sector_code"]:
    df[col] = pd.to_numeric(df[col], errors="raise").astype("Int64")

for col in ["emissions_tons", "real_gdp_millions_2017usd", "population"]:
    df[col] = pd.to_numeric(df[col], errors="raise")

display(df.dtypes.rename("cleaned_dtype"))
df = df.drop_duplicates().copy()

# Assert that no strings are missing (after removing whitespace)
assert df[["state_fips", "state", "year", "sector_code", "pollutant"]].notna().all().all(), "Missing identifiers; review before filtering."


In [ ]:
# Check missing emissions in sectors 0 and 99 before filtering sectors.
# Include observed years with zero missing values.
sector_missing_check = (
    df.loc[df["sector_code"].isin([0, 99])]
    .groupby(["sector_code", "year"])["emissions_tons"]
    .agg(observations="size", missing=lambda x: x.isna().sum())
)
display(sector_missing_check)

In [ ]:
# Match missing emissions to the same state, sector, and pollutant exactly 3 years earlier.
keys = ["state_fips", "sector_code", "pollutant", "year"]

prior = df[keys + ["emissions_tons"]].copy()
prior["year"] += 3
prior = prior.rename(columns={"emissions_tons": "emissions_3yrs_before"})

missing_check = df.loc[df["emissions_tons"].isna()].merge(
    prior,
    on=keys,
    how="left",
    validate="many_to_one",
)

# Unknown if the earlier observation is absent or its emissions are missing.
missing_check["below_10_3yrs_before"] = (
    missing_check["emissions_3yrs_before"].astype("Float64") < 30
)

display(missing_check[
    ["state", "sector", "pollutant", "year",
     "emissions_3yrs_before", "below_10_3yrs_before"]
])
display(missing_check["below_10_3yrs_before"].value_counts(dropna=False))


## 2. Keep the requested sector total and years

Inspect sector labels before filtering. Missing emissions in discarded component sectors do not affect the sector-0 and sector-99 totals. 

In [ ]:
display(df[["sector_code", "sector"]].drop_duplicates().sort_values("sector_code"))

# Keep the sector total and restrict the reporting period.
clean = df.loc[df["sector_code"].isin([0, 99])].copy()
clean = clean.loc[clean["year"].between(2005, 2023)].copy()

# Keep only the 50 states and DC; exclude any national or regional aggregate rows.
state_abbreviations = set("AL AK AZ AR CA CO CT DE DC FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY".split())
print("Excluded geographic rows:", (~clean["state"].isin(state_abbreviations)).sum())
clean = clean.loc[clean["state"].isin(state_abbreviations)].copy()
display(clean.isna().sum().rename("missing_values_in_retained_rows"))

# Each state abbreviation must identify exactly one FIPS code, and vice versa.
assert clean.groupby("state")["state_fips"].nunique().eq(1).all()
assert clean.groupby("state_fips")["state"].nunique().eq(1).all()


Export the granular data for the analysis around the great recession here.

In [ ]:
recession_data = df.loc[df["sector_code"].isin([0, 1])].copy()
recession_data = recession_data.loc[recession_data["year"].between(2008, 2011)].copy()
# Keep only the 50 states and DC; exclude any national or regional aggregate rows.
print("Excluded geographic rows:", (~recession_data["state"].isin(state_abbreviations)).sum())
recession_data = recession_data.loc[recession_data["state"].isin(state_abbreviations)].copy()
display(recession_data.isna().sum().rename("missing_values_in_retained_rows"))

## 3. Check the reporting levels and coverage

Check emissions separately for sectors 0 and 99. Each year, sector, and pollutant must have nonmissing emissions for all 50 states plus DC. Count GDP once per state-year across both sectors and all pollutants. No interpolation or zero filling is applied.


In [ ]:
emissions_key = ["state_fips", "year", "sector_code", "pollutant"]
gdp_key = ["state_fips", "year"]
gdp_col = "real_gdp_millions_2017usd"

# Keep one emissions value per state-year-sector-pollutant.
emissions_conflicts = clean.groupby(emissions_key)["emissions_tons"].nunique(dropna=False)
assert emissions_conflicts.le(1).all(), "Conflicting emissions for a state-year-sector-pollutant."
state_emissions = clean[emissions_key + ["emissions_tons"]].drop_duplicates(emissions_key)
assert state_emissions["emissions_tons"].notna().all(), "Missing emissions in sector 0 or 99."
assert set(state_emissions["sector_code"].unique()) == {0, 99}, "Expected both sectors 0 and 99."

# GDP is repeated across sectors and pollutants: keep it once per state-year.
gdp_conflicts = clean.groupby(gdp_key)[gdp_col].nunique(dropna=False)
assert gdp_conflicts.le(1).all(), "Conflicting GDP values for a state-year."
state_gdp = clean[gdp_key + [gdp_col]].drop_duplicates(gdp_key)
assert state_gdp[gdp_col].notna().all(), "Missing state-year GDP."

coverage = (
    state_emissions.groupby(["year", "sector_code", "pollutant"])["state_fips"]
    .nunique()
    .unstack(["sector_code", "pollutant"])
)
display(coverage)
assert coverage.eq(51).all().all(), "Incomplete state coverage by year, sector, or pollutant."
assert state_gdp.groupby("year")["state_fips"].nunique().eq(51).all(), "Incomplete GDP coverage."

available_years = sorted(clean["year"].unique().tolist())
print("Reporting years:", available_years)

## 4. Aggregate national annual totals

Sum emissions separately for sectors 0 and 99 and display them side by side, in tons. GDP is in millions of 2017 US dollars.


In [ ]:
national_annual_emissions = (
    state_emissions.groupby(["year", "pollutant", "sector_code"])["emissions_tons"]
    .sum(min_count=1)
    .unstack("sector_code")
    .rename(columns={0: "sector_0_emissions_tons", 99: "sector_99_emissions_tons"})
    .rename_axis(columns=None)
    .reset_index()
)

national_annual_real_gdp = (
    state_gdp.groupby("year", as_index=False)[gdp_col]
    .sum(min_count=1)
    .sort_values("year")
    .reset_index(drop=True)
)

display(national_annual_emissions)
display(national_annual_real_gdp)


## 5. Save cleaned datasets for reuse

Save Parquet files beside the input data. Both exports cover the retained reporting years (2005–2023), include the 50 states and DC, and use only the existing sector totals 0 and 99. These are separate alternatives and are never summed together.

- `NEI_clean_state.parquet`: one row per state, year, and pollutant, with separate emissions columns for sectors 0 and 99, plus GDP and population. Sector codes and sector labels are omitted.
- `NEI_clean_national.parquet`: one row per year and pollutant, with separate emissions columns for sectors 0 and 99 and national real GDP.


In [ ]:
# Export the cleaned state observations and the national totals.
output_dir = csv_path.parent
# Keep state-year attributes once, then pivot the two sector alternatives.
state_attributes = ["state", gdp_col, "population"]
assert clean.groupby(gdp_key)[state_attributes].nunique(dropna=False).le(1).all().all(), "Conflicting state-year attributes."
state_details = clean[gdp_key + state_attributes].drop_duplicates(gdp_key)
clean_state = (
    state_emissions.pivot(
        index=["state_fips", "year", "pollutant"],
        columns="sector_code",
        values="emissions_tons",
    )
    .rename(columns={0: "sector_0_emissions_tons", 99: "sector_99_emissions_tons"})
    .rename_axis(columns=None)
    .reset_index()
    .merge(state_details, on=gdp_key, how="left", validate="many_to_one")
    .sort_values(["state_fips", "year", "pollutant"])
    .reset_index(drop=True)
)
clean_state = clean_state[[
    "state_fips", "state", "year", "pollutant",
    "sector_0_emissions_tons", "sector_99_emissions_tons",
    gdp_col, "population",
]]
assert not clean_state.duplicated(["state_fips", "year", "pollutant"]).any()
assert clean_state[["sector_0_emissions_tons", "sector_99_emissions_tons"]].notna().all().all()

clean_national = national_annual_emissions.merge(
    national_annual_real_gdp,
    on="year",
    how="left",
    validate="many_to_one",
).sort_values(["year", "pollutant"]).reset_index(drop=True)
assert not clean_national.duplicated(["year", "pollutant"]).any()
assert clean_national.notna().all().all(), "Missing national export values."

state_output = output_dir / "NEI_clean_state.parquet"
national_output = output_dir / "NEI_clean_national.parquet"
clean_state.to_parquet(state_output, index=False, engine="pyarrow")
clean_national.to_parquet(national_output, index=False, engine="pyarrow")
print(f"Saved {len(clean_state):,} rows to {state_output.resolve()}")
print(f"Saved {len(clean_national):,} rows to {national_output.resolve()}")


# Clean recession data
assert recession_data.groupby(gdp_key)[state_attributes].nunique(dropna=False).le(1).all().all(), "Conflicting state-year attributes."
recession_emissions = recession_data[emissions_key + ["emissions_tons"]].drop_duplicates(emissions_key)
state_details_recession = recession_data[gdp_key + state_attributes].drop_duplicates(gdp_key)
clean_recession = (
    recession_emissions.pivot(
        index=["state_fips", "year", "pollutant"],
        columns="sector_code",
        values="emissions_tons",
    )
    .rename(columns={0: "sector_0_emissions_tons", 1: "sector_1_emissions_tons"})
    .rename_axis(columns=None)
    .reset_index()
    .merge(state_details_recession, on=gdp_key, how="left", validate="many_to_one")
    .sort_values(["state_fips", "year", "pollutant"])
    .reset_index(drop=True)
)
clean_recession = clean_recession[[
    "state_fips", "state", "year", "pollutant",
    "sector_0_emissions_tons", "sector_1_emissions_tons",
    gdp_col, "population",
]]
recession_output = output_dir / "NEI_clean_recession.parquet"
clean_recession.to_parquet(recession_output, index=False, engine="pyarrow")
print(f"Saved {len(clean_recession):,} rows to {recession_output.resolve()}")



## 6. Plot emissions and real GDP (2005 = 100)

Create two separate graphs: one for sector 0 and one for sector 99, each including real GDP. Each pollutant and sector series uses its own 2005 baseline. Pollutant colors and markers and the axis limits are consistent across both graphs; GDP is black. Edit `graph_notes` below and rerun the plotting cell to update the notes beneath both graphs. Each graph is also saved as a separate PNG.


In [ ]:
import matplotlib.pyplot as plt
from textwrap import fill

# One column for each pollutant-sector series, plus GDP counted only once.
emissions_series = national_annual_emissions.pivot(
    index="year", columns="pollutant",
    values=["sector_0_emissions_tons", "sector_99_emissions_tons"],
).sort_index()
gdp_series = national_annual_real_gdp.set_index("year")[gdp_col].sort_index()
assert 2005 in emissions_series.index and 2005 in gdp_series.index, "Missing 2005 baseline."
assert emissions_series.loc[2005].notna().all() and emissions_series.loc[2005].gt(0).all(), "Emissions baselines must be positive and nonmissing."
assert pd.notna(gdp_series.loc[2005]) and gdp_series.loc[2005] > 0, "GDP baseline must be positive and nonmissing."
emissions_index = emissions_series.div(emissions_series.loc[2005]).mul(100)
gdp_index = gdp_series.div(gdp_series.loc[2005]).mul(100)

pollutants = sorted(national_annual_emissions["pollutant"].unique())
pollutant_labels = {"NOX": r"$NO_X$", "PM25": r"$PM_{2.5}$", "SO2": r"$SO_2$", "VOC": r"$VOCs$"}
colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]
markers = ["o", "s", "^", "D", "v", "P", "X", "<"]
# Use the same scale so the two alternatives are easy to compare.
y_max = max(emissions_index.max().max(), gdp_index.max()) * 1.08
figures = {}
for sector in [0, 99]:
    fig, ax = plt.subplots(figsize=(7.5, 5))
    # Leave room below the x-axis label for the legend and editable notes.
    fig.subplots_adjust(bottom=0.31, top=0.9)
    for i, pollutant in enumerate(pollutants):
        series = emissions_index[(f"sector_{sector}_emissions_tons", pollutant)]
        ax.plot(
            series.index, series,
            color=colors[i], marker=markers[i],
            linestyle="-", linewidth=1.7, markersize=6,
            label=pollutant_labels.get(pollutant, pollutant),
        )
    ax.plot(gdp_index.index, gdp_index, color="black", marker="*",
            linewidth=2.3, markersize=10, label="Real GDP (2017$)")
    ax.set_ylabel(" 2005 = 100 ")
    ax.set_ylim(0, y_max)
    ax.set_xticks(sorted(set(emissions_index.index) | set(gdp_index.index)))
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    fig.legend(*ax.get_legend_handles_labels(), loc="lower center",
               bbox_to_anchor=(0.5, 0.105), ncol=3, frameon=False)
    figures[sector] = fig
    fig.savefig(f"emissions_gdp_sector_{sector}.png", dpi=200, bbox_inches="tight")
    plt.show()
